# Dynamic Reflections: video-text alignment on Colab (one run)

**Before you start (2 minutes, once)**
1. Colab left sidebar, key icon (Secrets): add a secret named `HF_TOKEN` with your Hugging Face *Read* token and switch on **Notebook access**.
   Accept the Gemma license at https://huggingface.co/google/gemma-2-2b-it (same account as the token).
2. Runtime > Change runtime type > **T4 GPU**.
3. Runtime > **Run all**. The only interaction is the Google Drive permission pop-up at the start.

**What it does, in order** (each stage is reported at the end; a failed stage never wastes the ones that can still run)
- Cheap checks first: Hugging Face access, disk, GPU.
- Downloads the ~1K PVD videos (resumable, stored on Drive).
- Builds the environment, then a **smoke test on 8 videos** exercising every model and code path, so any problem shows up in minutes, before the long steps.
- Baseline alignment score, frames x captions sweeps with the scaling-law fit, encoder comparison, video-to-text retrieval demo.
- Releases the GPU runtime when finished (or after a critical failure) so no compute units are wasted. Set `DISCONNECT_WHEN_DONE = False` below to keep it.

**To save GPU units:** run this notebook once on a *CPU* runtime first. It only prepares data (downloads the videos to Drive), which costs no GPU. Then switch to T4 and Run all; the download is skipped because it is already on Drive.

Text model: Gemma-2-2B-it (the paper uses 9B), so absolute scores will be lower; compare trends. Every score is printed with N, k and the chance level.

## 1. Settings, Drive, project code, Hugging Face token

In [ ]:
# ===== settings =====
DISCONNECT_WHEN_DONE = True    # release the runtime at the end / after a critical failure (stops GPU billing)
SMOKE_VIDEOS = 8               # size of the quick end-to-end test that runs before the long steps
CAPTION_COUNTS = [1, 2, 4, 10]                       # captions per video to sweep
FRAME_GRIDS = {'dinov2_large_video': [1, 2, 4, 8, 16],   # frames per video to sweep, per encoder
               'videomaev2_base': [16, 32, 48]}          # (VideoMAEv2 clip = 16 frames)

import os, sys, subprocess
from pathlib import Path
BASE = Path(os.environ.get('GENAI_BASE', '/content'))    # only changed by the local test harness
from google.colab import drive
drive.mount(str(BASE / 'drive'))
DRIVE_DIR = BASE / 'drive' / 'MyDrive' / 'GenAI_Project'
PROJECT = DRIVE_DIR / 'dynamic-reflections-vtr'

if not PROJECT.exists():
    PROJECT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', 'https://github.com/Rhytam23/dynamic-reflections-vtr.git', str(PROJECT)], check=True)
else:  # cloned in an earlier session: pick up the latest fixes
    r = subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'])
    if r.returncode != 0:
        print('WARNING: git pull failed; using the code already on Drive')
sys.path.insert(0, str(PROJECT))

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    pass
if not HF_TOKEN:  # no secret: fall back to the interactive login box
    from huggingface_hub import notebook_login, get_token
    notebook_login()
    HF_TOKEN = get_token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN   # inherited by the feature-extraction subprocesses

## 2. Imports and stage runner

In [ ]:
import json, logging, shutil
import numpy as np
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s', force=True)
from refactored_modules import features, pipeline, preflight, sweeps, utils, plots, retrieval, colab_inference
from refactored_modules.alignment import layer_sweep
from refactored_modules.config import PVDConfig
from refactored_modules.runner import StageRunner, SkipStage

config = PVDConfig(drive_base_dir=DRIVE_DIR, work_dir=BASE, caption_counts=CAPTION_COUNTS)
REPO = config.repo_full_path                 # authors' repo + venv on local disk
FEAT = REPO / config.feature_dir             # features (on Drive through a symlink)
RES = REPO / config.results_dir              # our results and figures (on Drive)
ANN = REPO / config.dataset_json_path

run = StageRunner()
S_PRE, S_GPU, S_REPO, S_DATA, S_ENV = 'Preflight: token + disk', 'GPU', 'Authors repo', 'PVD videos', 'Environment'
S_SMOKE, S_VMAE, S_BASE, S_CAP = 'Smoke test', 'Smoke test: VideoMAEv2', 'Baseline alignment', 'Sweep: caption features'
S_REPORT, S_RETR, S_DEMO, S_DESC = 'Report', 'Retrieval', 'Retrieval demo', 'Description (optional)'
S_SWEEP = lambda m: f'Sweep: {m}'

## 3. Cheap checks and data preparation (no GPU needed)

In [ ]:
@run.stage(S_PRE, critical=True)
def _():
    print('Hugging Face access OK for:', preflight.check_hf_access(HF_TOKEN))
    print('free disk (GB):', round(preflight.check_disk(BASE, 25)))

@run.stage(S_GPU)
def _():
    if not preflight.has_gpu():
        raise SkipStage('CPU runtime: only data preparation runs. Switch to a T4 GPU runtime and Run all again.')
    print('GPU:', preflight.check_gpu())

@run.stage(S_REPO, critical=True, needs=[S_PRE])
def _():
    utils.clone_authors_repo(REPO, config.persist_dir)
    utils.patch_scripts(REPO)

@run.stage(S_DATA, critical=True, needs=[S_REPO])
def _():
    pipeline.download_pvd_dataset(config.dataset_json_path, config.dataset_output_dir, repo_path=REPO)
    n = preflight.check_videos_present(ANN, REPO / config.dataset_output_dir)
    print(n, 'videos ready in', REPO / config.dataset_output_dir)
    return n

## 4. Environment and smoke test (8 videos, every model and code path)

In [ ]:
@run.stage(S_ENV, critical=True, needs=[S_GPU, S_REPO])
def _():
    utils.setup_environment(REPO, config.persist_dir)

@run.stage(S_SMOKE, critical=True, needs=[S_ENV, S_DATA])
def _():
    shutil.rmtree(REPO / 'results' / 'smoke', ignore_errors=True)   # never trust files from an earlier failed try
    preflight.make_smoke_annotation(ANN, REPO / 'assets' / 'smoke.jsonl', SMOKE_VIDEOS)
    kw = dict(feature_dir=Path('results/smoke'), annotation_path=Path('assets/smoke.jsonl'))
    features.extract_features(REPO, 'llm', llm_names=[config.llm_name], hf_token=HF_TOKEN, **kw)
    features.extract_features(REPO, 'llm', llm_names=[config.llm_name], num_captions=2, hf_token=HF_TOKEN, **kw)
    features.extract_features(REPO, 'video', video_names=[config.vision_name], **kw)
    features.extract_features(REPO, 'video', video_names=[config.vision_name], num_frames=4, **kw)
    d = REPO / 'results' / 'smoke'
    llm = features.feature_path(d, 'pvd', config.llm_name, config.llm_pool)
    vis = features.feature_path(d, 'pvd', config.vision_name, config.vision_pool)
    for path in (llm, vis,
                 features.feature_path(d, 'pvd', config.llm_name, config.llm_pool, num_captions=2),
                 features.feature_path(d, 'pvd', config.vision_name, config.vision_pool, num_frames=4)):
        print(path.name, preflight.check_feature_file(path, SMOKE_VIDEOS))
    _, best = layer_sweep(features.load_features(vis), features.load_features(llm), k=3)
    print('alignment code runs on real features (score is meaningless with 8 videos):', best)

@run.stage(S_VMAE, needs=[S_SMOKE])
def _():
    kw = dict(feature_dir=Path('results/smoke'), annotation_path=Path('assets/smoke.jsonl'))
    features.extract_features(REPO, 'video', video_names=['videomaev2_base'], **kw)
    path = features.feature_path(REPO / 'results' / 'smoke', 'pvd', 'videomaev2_base', config.pool_for('videomaev2_base'))
    print(path.name, preflight.check_feature_file(path, SMOKE_VIDEOS))

## 5. Baseline alignment score
Gemma-2-2B-it text features (all 10 captions) and DINOv2-large video features (default frames), then the layer x layer mutual k-NN sweep.

In [ ]:
@run.stage(S_BASE, needs=[S_SMOKE])
def _():
    features.extract_features(REPO, 'llm', llm_names=[config.llm_name], hf_token=HF_TOKEN)
    features.extract_features(REPO, 'video', video_names=[config.vision_name])
    n = run.results[S_DATA]
    for path in (features.feature_path(FEAT, 'pvd', config.llm_name, config.llm_pool),
                 features.feature_path(FEAT, 'pvd', config.vision_name, config.vision_pool)):
        print(path.name, preflight.check_feature_file(path, n))
    result = pipeline.run_alignment_pipeline(FEAT, config.llm_name, config.vision_name, k=config.k,
                                             output_json=RES / 'baseline.json')
    print(json.dumps(result, indent=2))
    scores = np.load(RES / 'baseline.layers.npy')
    best = {'layer_x': result['best_vision_layer'], 'layer_y': result['best_llm_layer'],
            'score': result['mutual_knn_score'], 'n': result['n'], 'k': result['k']}
    plots.plot_layer_heatmap(scores, best, RES / 'figures' / 'layer_heatmap.png')
    return result

## 6. Sweeps: frames x captions, scaling-law fit, encoder comparison
Extraction is resumable (finished feature files are skipped). Each encoder is its own stage, so if VideoMAEv2 fails DINOv2 still completes.

In [ ]:
@run.stage(S_CAP, needs=[S_BASE])
def _():
    n_cap = preflight.caption_counts_uniform(ANN)
    if n_cap in CAPTION_COUNTS:  # the baseline already ran on all captions: reuse it for that count
        print(f'every video has {n_cap} captions: reusing the baseline LLM run for count {n_cap}')
        sweeps.alias_full_caption_features(FEAT, config.llm_name, n_cap, config.llm_pool)
    sweeps.extract_caption_sweep(REPO, config.feature_dir, config.llm_name, CAPTION_COUNTS, HF_TOKEN)

for model in config.video_models:
    @run.stage(S_SWEEP(model), needs=[S_CAP] + ([S_VMAE] if model.startswith('videomae') else []))
    def _(model=model):
        frames = FRAME_GRIDS[model]
        sweeps.extract_frame_sweep(REPO, config.feature_dir, model, frames)
        res = sweeps.fit_grid(sweeps.compute_grid(FEAT, config.llm_name, model, frames, CAPTION_COUNTS,
                                                  k=config.k, vision_pool=config.pool_for(model)))
        sweeps.save_result(res, RES / f'sweep_{model}.json')
        print(json.dumps({k: res[k] for k in ('vision', 'n', 'k', 'chance_level', 'fit')}, indent=2))
        return res

@run.stage(S_REPORT)
def _():
    done = [run.results[S_SWEEP(m)] for m in config.video_models if run.status.get(S_SWEEP(m)) == 'ok']
    if not done:
        raise SkipStage('no sweep finished')
    table = sweeps.comparison_table(done)
    print(table)
    (RES / 'comparison.md').write_text(table + '\n')
    plots.plot_sweeps(done, RES / 'figures' / 'sweeps.png')
    for r in done:
        if r.get('fit'):
            plots.plot_fit(r, RES / 'figures' / f"fit_{r['vision']}.png")
    from IPython.display import Image, display
    display(Image(str(RES / 'figures' / 'sweeps.png')))
    print('For reference, the paper (Gemma-2-9B) reports C_f = 0.15 for VideoMAEv2 vs 0.05 for DINOv2.')

## 7. Video-to-text demo
Held-out videos (20%) are matched to captions. The spaces differ in size, so a map is learned from the training pairs (ridge), or each item is described
by its similarity to the training items (relative). Layers are chosen on the training split only. **Uses paired training data, so it is not zero-shot.**

In [ ]:
@run.stage(S_RETR, needs=[S_BASE])
def _():
    samples = [json.loads(l) for l in ANN.read_text().splitlines() if l.strip()]
    v = features.load_features(features.feature_path(FEAT, 'pvd', config.vision_name, config.vision_pool))
    t = features.load_features(features.feature_path(FEAT, 'pvd', config.llm_name, config.llm_pool))
    assert len(samples) == len(v) == len(t), 'feature order must match the annotation file'
    ev = retrieval.evaluate_retrieval(v, t, k_align=config.k)
    summary = {k: ev[k] for k in ev if k not in ('_sims', 'test_indices')}
    print(json.dumps(summary, indent=2))
    (RES / 'retrieval.json').write_text(json.dumps(summary, indent=2))
    return {'ev': ev, 'samples': samples}

@run.stage(S_DEMO, needs=[S_RETR])
def _():
    import matplotlib.pyplot as plt
    ev, samples = run.results[S_RETR]['ev'], run.results[S_RETR]['samples']
    first_caption = [s['rephrased_captions'][0] for s in samples]   # one caption per gallery video
    test_idx, sim, examples = ev['test_indices'], ev['_sims']['ridge'], []
    for row in range(3):
        vid = samples[test_idx[row]]
        retrieved = [first_caption[test_idx[j]] for j in sim[row].argsort()[::-1][:3]]
        frames = colab_inference.sample_frames(REPO / config.dataset_output_dir / f"{vid['video_id']}.bin", 5)
        fig, axes = plt.subplots(1, len(frames), figsize=(16, 3))
        for ax, fr in zip(axes, frames):
            ax.imshow(fr); ax.axis('off')
        plt.show()
        print('True caption:', first_caption[test_idx[row]])
        print('Retrieved   :', retrieved, '\n')
        examples.append((first_caption[test_idx[row]], retrieved))
    return examples

@run.stage(S_DESC, needs=[S_DEMO])
def _():
    examples = run.results[S_DEMO]
    for (true, _r), text in zip(examples, colab_inference.generate_descriptions([r for _t, r in examples])):
        print('True :', true)
        print('Fused:', text, '\n')

## 8. Summary (and release the GPU)

In [ ]:
run.finish(disconnect=DISCONNECT_WHEN_DONE, summary_path=DRIVE_DIR / 'run_summary.json')